In [ ]:
import requests
from bs4 import BeautifulSoup
import re
from tabulate import tabulate
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.units import cm
from reportlab.lib.utils import ImageReader
from reportlab.platypus import Paragraph, Frame
from reportlab.lib.styles import getSampleStyleSheet
import os

def species_info(species):
    headers = {"User-Agent":""}
    url="https://waarnemingen.be/species/search/"
    wiki_url=f"https://en.wikipedia.org/wiki/{species}"
    search_parameters={"q":species,"species_group":16}
    response = requests.get(url,params=search_parameters)
    rarity_soup = BeautifulSoup(response.text,"html.parser")
    rarity_status_dutch = rarity_soup.find("span",class_="hidden-sm").get_text(strip=True)
    translation_api = "https://api.mymemory.translated.net/get"
    params = {
        "q": rarity_status_dutch,
        "langpair": "nl|en"
    }
    r = requests.get(translation_api, params=params, timeout=10)
    data = r.json()
    rarity_status_english = data["responseData"]["translatedText"]
    soup = BeautifulSoup(response.text,"html.parser")
    new_url = soup.find("a",href=re.compile(r"^/species/\d+/$"))
    if new_url is None:
        return "This species doesnot belong to Coleoptera,Thank you"
    species_url="https://waarnemingen.be"+new_url["href"]
    response2=requests.get(species_url)
    soup2=BeautifulSoup(response2.text,"html.parser")
    description_in_dutch = soup2.find("div",class_="app-content-section js-user-content").get_text(strip=True)
    response3=requests.get(wiki_url,headers=headers)
    soup3=BeautifulSoup(response3.text,"html.parser")
    div_container=soup3.find("div",class_="mw-content-ltr mw-parser-output")
    para=""
    for i in div_container.find_all("p"):
        if i.get_text(strip=True):
            para=i
            break
    for j in para.find_all("sup"):
        j.decompose()
    for k in para.find_all("style"):
        k.decompose()
    description_in_english=para.get_text()

    namur_url="https://waarnemingen.be"+new_url["href"]+"observations/?date_after=2024-12-25&date_before=2025-12-25&country_division=24&search=&user=&location=&sex=&month=&life_stage=&activity=&method="
    response4=requests.get(namur_url)
    soup4=BeautifulSoup(response4.text,"html.parser")
    table = soup4.find("table",class_="table table-bordered table-striped")
    lines = table.find("tbody").find_all("tr")
    data = []
    for i in lines:
        obs=i.find_all("td")
        if len(obs)<3:
            continue
        date = obs[0].get_text(strip=True)
        rawnumber = obs[1].get_text(" ", strip=True)
        match = re.search(r"\d+", rawnumber)
        number = match.group()
        location = obs[2].get_text(strip=True)
        data.append([date,number,location])
        if len(data)==10:
            break
        
    species_img = None
    img_tags = soup2.find_all("img")
    img_url = None
    for i in img_tags:
        src=i.get("src")
        if src.startswith("/media/photo/"):
            img_url="https://waarnemingen.be"+src
            break

    if img_url:
        response_img=requests.get(img_url)
        if response_img.status_code==200:
            species_img = f"{species}_img.jpg"
            with open(species_img,"wb") as f:
                for pic in response_img.iter_content(1024):
                    f.write(pic)
    
    full_species_informations = {
        "Name":species,
        "Latin Name":species,
        "Rarity Status(Dutch)":rarity_status_dutch,
        "Rarity Status(english)":rarity_status_english,
        "Description(Dutch)":description_in_dutch,
        "Description(English)":description_in_english,
        "Observation of the species in Namur": data,
        "image_path": species_img
        
    }

    return full_species_informations

def draw_page_border(c, width, height, margin=1.5*cm):
    c.setLineWidth(1)
    c.rect(
        margin,
        margin,
        width - 2*margin,
        height - 2*margin
    )

def generate_species_pdf(
    species,
    latin_name,
    rarity_status_dutch,
    rarity_status_english,
    description_dutch,
    description_english,
    observations,
    image_path
):
    pdf_file = f"{species}_report.pdf"
    c = canvas.Canvas(pdf_file, pagesize=A4)
    width, height = A4

    margin = 1.5 * cm
    c.setLineWidth(1)
    c.rect(
        margin,
        margin,
        width - 2 * margin,
        height - 2 * margin
    )

    styles = getSampleStyleSheet()
    normal = styles["Normal"]
    heading = styles["Heading2"]

    header_top = height - margin - 0.5 * cm
    text_x = margin + 0.5 * cm

    c.setFont("Helvetica-Bold", 16)
    c.drawString(text_x, header_top, species)

    c.setFont("Helvetica", 11)
    c.drawString(text_x, header_top - 1.0 * cm, f"Latin name: {latin_name}")
    c.drawString(
        text_x,
        header_top - 1.7 * cm,
        f"Rarity (NL): {rarity_status_dutch}"
    )
    c.drawString(
        text_x,
        header_top - 2.4 * cm,
        f"Rarity (EN): {rarity_status_english}"
    )

    image_box = 4.5 * cm
    if image_path and os.path.exists(image_path):
        try:
            img = ImageReader(image_path)
            c.drawImage(
                img,
                width - margin - image_box,
                header_top - image_box,
                width=image_box,
                height=image_box,
                preserveAspectRatio=True,
                mask="auto"
            )
        except Exception:
            pass

    header_height = 5.5 * cm  
    frame = Frame(
        margin + 0.5 * cm,
        margin + 0.5 * cm,
        width - 2 * margin - 1 * cm,
        height - header_height - margin,
        showBoundary=0
    )

    story = []

    story.append(Paragraph("<b>Description (English)</b>", heading))
    story.append(Paragraph(description_english, normal))
    story.append(Paragraph("<br/>", normal))

    story.append(Paragraph("<b>Description (Dutch)</b>", heading))
    story.append(Paragraph(description_dutch, normal))
    story.append(Paragraph("<br/>", normal))

    story.append(Paragraph("<b>Recent observations in Namur</b>", heading))

    if observations:
        for date, number, location in observations:
            story.append(
                Paragraph(
                    f"<b>{date}</b> — {number} observed at {location}",
                    normal
                )
            )
    else:
        story.append(Paragraph("No observations available.", normal))

    frame.addFromList(story, c)

    c.save()
    print(f"PDF generated: {pdf_file}")
    return {
    "pdf_file": pdf_file,
    "species": species,
    "image_used": image_path,
    "observations_count": len(observations) if observations else 0
    }

if __name__ == "__main__":
    info=species_info("Laccophilus minutus")
    generate_species_pdf(
    species=info["Name"],
    latin_name=info["Latin Name"],
    rarity_status_dutch=info["Rarity Status(Dutch)"],
    rarity_status_english=info["Rarity Status(english)"],
    description_dutch=info["Description(Dutch)"],
    description_english=info["Description(English)"],
    observations=info["Observation of the species in Namur"],
    image_path=info["image_path"]
)

PDF generated: Laccophilus minutus_report.pdf


In [17]:
pip install folium

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from bs4 import BeautifulSoup
import requests
import re
import folium
import time
import datetime
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from folium.plugins import MarkerCluster
import random

def observations_map(day=None):
    if day is None:
        day = datetime.datetime.now().strftime("%Y-%m-%d")
    
    url = f"https://waarnemingen.be/fieldwork/observations/daylist/?date={day}&species_group=16&country_division=24&rarity=&search="
    response = requests.get(url)
    soup=BeautifulSoup(response.text,"html.parser")
    data_table = soup.find("table",class_="table table-bordered table-striped")
    if not data_table:
        print("opps..No observation found")
        return
    data_rows = data_table.find("tbody").find_all("tr")
    locator = Nominatim(user_agent="sadman")
    geocode = RateLimiter(locator.geocode)
    X = folium.Map(location=[50.4674,4.8718],zoom_start=9)
    cluster = MarkerCluster().add_to(X)
    color_for_species = {}

    for i in data_rows:
        data_col=i.find_all("td")
        if len(data_col)<5:
            continue
        species_namess=data_col[3]
        Common_name = species_namess.find("span", class_="species-common-name")
        scientific_name=species_namess.find("i",class_="species-scientific-name")
        if Common_name:
            name=Common_name.get_text(strip=True)
        elif scientific_name:
            name=scientific_name.get_text(strip=True)
        else:
            continue

        location = data_col[4]
        location_name=location.find_all("a")
        for i in location_name:
            raw_location=i.get_text(strip=True)
            clean_location=raw_location.split("(")[0].split(",")[0].strip()
            if name not in color_for_species:
                color_for_species[name] = "#{:06x}".format(random.randint(0, 0xFFFFFF))
            realLocation=geocode(clean_location+ ",Namur,Belgium")
            if not realLocation:
                continue
            folium.Marker(location=[realLocation.latitude,realLocation.longitude],popup=f"<b>{name}</b><br>{raw_location}</br>{day}",icon=folium.Icon(color=color_for_species[name],icon="smile",prefix="fa")).add_to(cluster)

    legend_html = """
        <div style="
            position: fixed;
            bottom: 50px;
            left: 50px;
            z-index: 9999;
            background-color: white;
            padding: 10px;
            border: 2px solid grey;
            font-size: 14px;
        ">
        <b>Species names</b><br>
        """

    for species, color in color_for_species.items():
            legend_html += f"""
            <i style="color:{color}">●</i> {species}<br>
            """

    legend_html += "</div>"
    X.get_root().html.add_child(folium.Element(legend_html))
    mapfilename = f"{day}.html"
    X.save(mapfilename)
    return {
    "date": day,
    "map_file": mapfilename,
    "species_count": len(color_for_species),
    "species_colors": color_for_species,
    "map_object": X
    }

observations_map("2025-04-18")


/var/folders/76/z1mjvk1s7k36fwnyfkcvx1q40000gn/T/ipykernel_1087/651938633.py:54: UserWarning:

color argument of Icon should be one of: {'lightgray', 'beige', 'darkpurple', 'darkblue', 'blue', 'darkgreen', 'red', 'orange', 'cadetblue', 'lightred', 'gray', 'white', 'black', 'darkred', 'green', 'pink', 'lightgreen', 'lightblue', 'purple'}.



{'date': '2025-04-18',
 'map_file': '2025-04-18.html',
 'species_count': 49,
 'species_colors': {'Gyrinus spec.': '#f259b3',
  'Gewone breedborst': '#32a8d0',
  'Echte breedborst': '#f1f9d1',
  'Groene zandloopkever': '#9188a5',
  'Kortnekloopkever onbekend': '#a9a5a0',
  'Bosspiegelloopkever': '#a1a0e5',
  'Kielsprietloopkever onbekend': '#b39a9b',
  'Rondhalszwartschild': '#1b9e8d',
  'Diksprietwaterroofkever': '#f20a47',
  'Agabus paludosus': '#5aeab9',
  'Gewone geelrand': '#15bf18',
  'Hydroglyphus geminus': '#8fe23b',
  'Hydroporus erythrocephalus': '#57a5f6',
  'Hydroporus incognitus': '#61e80f',
  'Moeraswaterroofkevertje': '#a08d1e',
  'Dwergwatertor': '#23e639',
  'Hygrotus impressopunctatus': '#3f24ba',
  'Eironde watertor': '#404f1b',
  'Ilybius spec.': '#dcfc1a',
  'Laccophilus minutus': '#53cd3e',
  'Grote spinnende watertor': '#3acaa5',
  'Harige kortschildkever': '#b44204',
  'Langvleugelmoskortschildkever': '#04ad88',
  'Platydracus chalcocephalus': '#c9440b',
  'Platy

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import json
import datetime
import calendar
import plotly.express as px

def seasonal_analysis(species,year):
    headers = {"User-Agent":""}
    url="https://waarnemingen.be/species/search/"
    search_parameters={"q":species,"species_group":16}
    response = requests.get(url,params=search_parameters)
    soup = BeautifulSoup(response.text,"html.parser")
    new_url = soup.find("a",href=re.compile(r"^/species/\d+/$"))
    if not new_url:
        print("No species found")
        return
    species_url="https://waarnemingen.be"+new_url["href"]
    observation_url=f"{species_url}observations/?"
    date_parameters = {"date_after": f"{year}-01-01","date_before":f"{year}-12-31","country_division":24}
    new_response = requests.get(observation_url,params=date_parameters)
    soup1=BeautifulSoup(new_response.text,"html.parser")
    data_table = soup1.find("table",class_="table table-bordered table-striped")
    if "Geen resultaten" in data_table.get_text():
        print(f"No observations done in {year} for {species} in Namur")
        return
    data_rows = data_table.find_all("tr")
    M_counts={i:0 for i in range(1,13)}
    seasons_with_months = {"Winter": [12, 1, 2],"Spring": [3, 4, 5],"Summer": [6, 7, 8],"Autumn": [9, 10, 11]}
    s_counts = {"Winter": 0,"Spring": 0,"Summer": 0,"Autumn": 0}

    for i in data_rows:
        data_cols=i.find_all("td")
        if not data_cols:
            continue
        date=data_cols[0].get_text(strip=True)[:10]
        realdate = datetime.datetime.strptime(date,"%Y-%m-%d")
        M_counts[realdate.month] += 1
    for i, j in seasons_with_months.items():
        for k in j:
            s_counts[i] += M_counts[k]
    
    months = list(M_counts.keys())
    count = list(M_counts.values())
    names_m=[]
    for i in months:
        names_m.append(calendar.month_abbr[i])
    
    fig_month = px.pie(
    names=names_m,
    values=count,
    title=f"Monthly observations of {species} ({year})"
    )
    fig_month.write_html(f"{species}_{year}_seasonal.html")
    fig_month.show()
    
    fig_season = px.bar(
    x=list(s_counts.keys()),
    y=list(s_counts.values()),
    labels={"x": "Season", "y": "Observations"},
    title=f"Seasonal observations of {species} ({year})"
    )

    fig_season.write_html(f"{species}_{year}_seasonal_bar.html")
    fig_season.show()
    low_season = min(s_counts,key=s_counts.get)
    explanations = explanation(species,low_season,year)
    return {
        "species": species,
        "year": int(year),
        "monthly_counts": M_counts,
        "seasonal_counts": s_counts,
        "lowest_season": low_season,
        "llm_explanation": explanations,
        "monthly_file": f"{species}_{year}_seasonal.html",
        "seasonal_file": f"{species}_{year}_seasonal_bar.html"
    }

#i am using ollama..a platform where i can install any llm locally..so that i dont have to pay for llm access cuz i mostly used the free trails before...i can use it ofr llm api
def explanation(species,low_season,year):
    text = f"Why {species} is less observed in the year of {year} during {low_season}.Explain it purely"
    url = "http://localhost:11434/api/generate"

    payload = {
    "model": "gemma3:1b",
    "prompt": text,
    "stream": False
    }
    r = requests.post(url, json=payload)
    data = r.json()
    explaination_data = data.get("response")
    return explaination_data
    

seasonal_analysis("Carabus violaceus","2024")
    

Okay, let's break down why Carabus violaceus (the "Blood-red Fairy Spider") seems to be less frequently observed in the winter months of 2024 compared to other seasons. It's a fascinating and complex topic, and here’s a purely scientific explanation, considering the likely factors:

**1. Reduced Photoperiod & Seasonal Transition:**

* **Decreased Sunlight:** The biggest factor is a significant reduction in sunlight hours during winter.  Carabus violaceus are primarily active during the warmer months (spring and summer) when sunlight levels are highest. Less sunlight means less time for them to emerge, reproduce, and actively hunt.
* **Temperature Drop:**  Winter typically sees a dramatic drop in temperatures, impacting the insects' metabolic rates. This slows down their activity and reduces their overall energy reserves, making them less likely to seek out feeding opportunities.

**2. Changes in Food Availability & Availability:**

* **Reduced Insect Populations:** Winter often sees a 

In [58]:
import requests
from bs4 import BeautifulSoup
import datetime
import csv
import calendar

def biodiversity_analysis(month=None, year=None):
    todays_date = datetime.date.today()
    if month is None:
        month = todays_date.month
    if year is None:
        year = todays_date.year
    
    MONTH_TO_NUMBER = {"january": 1,"february": 2,"march": 3,"april": 4,"may": 5,"june": 6,"july": 7,"august": 8,"september": 9,"october": 10,"november": 11,"december": 12}
    month_number = MONTH_TO_NUMBER[month.lower()]
    total_day_month = calendar.monthrange(int(year),month_number)[1]
    url = "https://waarnemingen.be/fieldwork/observations/daylist/"
    translation_cache = {}
    csv_rows = []
    total = 0
    species_counts = {} 
    for i in range(1,total_day_month+1):
        date = f"{year}-{month_number:02d}-{i:02d}"
        parameters = {"species_group":16,"country_division":24,"date":date}
        response = requests.get(url,params=parameters)
        soup = BeautifulSoup(response.text,"html.parser")
        data_table = soup.find("table",class_="table table-bordered table-striped")
        if "Geen resultaten" in data_table.get_text():
            continue
        data_row = data_table.find("tbody").find_all("tr")
        for i in data_row:
            data_cols = i.find_all("td")
            raw_name = data_cols[3].get_text()
            common_dutch = None
            scientific_name = None
            if " - " in raw_name:
                common_dutch, scientific_name = raw_name.split(" - ", 1)
            else:
                scientific_name = raw_name
            location = data_cols[4].get_text(strip=True)
            common_english = None
            if common_dutch:
                common_english = translation(common_dutch,translation_cache)
            if not common_english:
                common_english = scientific_name
            X = f"{common_english} ({scientific_name})"
            total += 1
            species_counts[X] = species_counts.get(X,0)+1
            csv_rows.append([
                date,
                common_dutch or "",
                common_english,
                scientific_name,
                location
            ])
    csv_file = f"biodiversity_{month}_{year}.csv"
    with open(csv_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "Date",
            "Common name (Dutch)",
            "Common name (English)",
            "Scientific name",
            "Location"
        ])
        writer.writerows(csv_rows)
    print(f"Species: {len(species_counts)} unique species")
    print(f"Total observations: {total}")
    print(f"CSV saved as: {csv_file}")

    return {
    "month": month,
    "year": int(year),
    "total_observations": total,
    "unique_species_count": len(species_counts),
    "species_counts": species_counts,
    "csv_file": csv_file
    }

    
def translation(english_name,cache):
    if not english_name:
        return None
    if english_name in cache:
        return cache[english_name]
    url = "https://api.mymemory.translated.net/get"
    params = {"q": english_name,"langpair": "nl|en"}
    e_response = requests.get(url,params=params) 
    data = e_response.json()["responseData"]["translatedText"]
    cache[english_name] = data
    return data




biodiversity_analysis("february","2023")


Species: 39 unique species
Total observations: 69
CSV saved as: biodiversity_february_2023.csv


{'month': 'february',
 'year': 2023,
 'total_observations': 69,
 'unique_species_count': 39,
 'species_counts': {' Phloiophilus edwardsii ( Phloiophilus edwardsii)': 1,
  'Giant Rooster (Timarcha tenebricosa)': 2,
  'Beetle unknown (Coleoptera indet.)': 3,
  'Earth flea unknown (Alticinae indet.)': 1,
  'Four-spotted Asian ladybug (Harmonia axyridis f. spectabilis)': 6,
  'Polka Dot Asian Ladybug (Harmonia axyridis f. succinea)': 3,
  ' Dorytomus spec. ( Dorytomus spec.)': 1,
  'Fire Boiler (Pyrrhidium sanguineum)': 1,
  'Rosemary Golden Rooster (Chrysolina americana)': 1,
  'Small wasp boktor (Clytus arietis)': 1,
  'Ant beetle (Thanasimus formicarius)': 1,
  'Snail bait beetle (Phosphuga atrata)': 2,
  'Two-spotted Asian ladybug (Harmonia axyridis f. conspicua)': 1,
  'Yellow-black ribbed boktor (Rhagium mordax)': 1,
  'engraver beetle (Ips typographus)': 1,
  'Garden Shale Bit (Carabus nemoralis)': 1,
  'seven-spotted ladybird (Coccinella septempunctata)': 7,
  'Four-spotted ladybug